# 循环神经网络RNN（上）

## 使用RNNcell

以下只展示forward运算

In [7]:
import torch

batch_size = 1
seq_len = 3
input_size = 4
hidden_size = 2

cell = torch.nn.RNNCell(
    input_size=input_size,
    hidden_size=hidden_size
)

dataset = torch.randn(seq_len, batch_size, input_size)
hidden = torch.zeros(batch_size, hidden_size) # 创建h0

for idx, input in enumerate(dataset):
    hidden = cell(input, hidden)
    print(hidden)

tensor([[0.4970, 0.8218]], grad_fn=<TanhBackward0>)
tensor([[0.6467, 0.9394]], grad_fn=<TanhBackward0>)
tensor([[-0.0227,  0.0973]], grad_fn=<TanhBackward0>)


## 使用RNN

以下只展示forward运算

In [8]:
import torch

batch_size = 1
seq_len = 3
input_size = 4
hidden_size = 2
num_layers = 1 # 用于在RNN中指定需要多少层

cell = torch.nn.RNN(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers
)

# 这里维度的顺序只能如下，因为RNN对于输入和隐层的维度输入默认如下
# 可以在RNN中设置batch_first=True，输入顺序变为batch_size, seq_len, input_size
inputs = torch.randn(seq_len, batch_size, input_size)
hidden = torch.zeros(num_layers, batch_size, hidden_size)

out, hidden = cell(inputs, hidden) # 不再需要循环

print('Output size:', out.shape)
print('Output:', out)
print('Hidden size:', hidden.shape)
print('Hidden:', hidden)

Output size: torch.Size([3, 1, 2])
Output: tensor([[[-0.9479,  0.9341]],

        [[-0.5100,  0.6000]],

        [[ 0.6073, -0.7474]]], grad_fn=<StackBackward0>)
Hidden size: torch.Size([1, 1, 2])
Hidden: tensor([[[ 0.6073, -0.7474]]], grad_fn=<StackBackward0>)


## Case: "hello" → "ohlol"

字符串的向量化：one-hot编码  

<img src="img/onehot.png" width="45%">

In [ ]:
# RNNcell实现
import torch

input_size = 4 # 输入字符有四种
hidden_size = 4 # 输出字符还是四种
batch_size = 1

idx2char = ['e','h','l','o']
x_data = [1,0,2,2,3] # hello
y_data = [3,1,2,3,2] # ohlol

one_hot_lookup = [
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,0],
    [0,0,0,1]
] # 创建边长为4的单位矩阵(此时还是列表)，对角线为1，其余为0
one_hot_lookup

[[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]

In [ ]:
x_one_hot = [one_hot_lookup[x] for x in x_data]

inputs = torch.Tensor(x_one_hot).view(-1, batch_size, input_size)
# 将batch_size, input_size控制，第一个维度自然就是seq_len
labels = torch.LongTensor(y_data).view(-1, 1)
# 注意这里是labels而非输出的hidden
# 1为batch_size

In [9]:
class Model(torch.nn.Module):
    def __init__(self,input_size,hidden_size,batch_size):
        super(Model, self).__init__()
        self.batch_size = batch_size
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.rnncell = torch.nn.RNNCell(input_size = self.input_size, 
                                        hidden_size = self.hidden_size)
    
    def forward(self,input,hidden):
        hidden = self.rnncell(input,hidden)
        return hidden

    def init_hidden(self): # 给定h0
        return torch.zeros(self.batch_size,self.hidden_size)

net = Model(input_size,hidden_size,batch_size)

In [10]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)

In [11]:
for epoch in range(15):
    loss = 0
    optimizer.zero_grad()
    hidden = net.init_hidden()
    print('预测str为',end='')
    for input,label in zip(inputs,labels):
        hidden = net(input,hidden)
        loss+=criterion(hidden,label)
        _,idx = hidden.max(dim = 1)
        print(idx2char[idx.item()],end='')
    loss.backward()
    optimizer.step()
    print(', epoch[%d/15] loss: %.4f' % (epoch+1,loss.item()))

预测str为ooooo, epoch[1/15] loss: 6.9476
预测str为oooll, epoch[2/15] loss: 5.4682
预测str为ollll, epoch[3/15] loss: 4.9307
预测str为ollll, epoch[4/15] loss: 4.6209
预测str为ollll, epoch[5/15] loss: 4.2198
预测str为ollll, epoch[6/15] loss: 3.7666
预测str为ohlll, epoch[7/15] loss: 3.4063
预测str为ohlol, epoch[8/15] loss: 3.1688
预测str为ohlol, epoch[9/15] loss: 2.9709
预测str为ohlol, epoch[10/15] loss: 2.7851
预测str为ohlol, epoch[11/15] loss: 2.6454
预测str为ohlol, epoch[12/15] loss: 2.5640
预测str为ohlol, epoch[13/15] loss: 2.5151
预测str为ohlol, epoch[14/15] loss: 2.4778
预测str为ohlol, epoch[15/15] loss: 2.4442


形状	示例	含义
(5,)	[3, 1, 2, 3, 2]	一维，5 个元素
(5, 1)	[[3], [1], [2], [3], [2]]	二维，5 行、1 列
(1, 5)	[[3, 1, 2, 3, 2]]	二维，1 行、5 列

In [ ]:
# 以下为RNN实现，网络修改如下
import torch
input_size = 4
hidden_size = 4
batch_size = 1
seq_len = 5
num_layers = 1 # 需要指定RNN层数

idx2char = ['e','h','l','o']
x_data = [1,0,2,2,3] # hello
y_data = [3,1,2,3,2] # ohlol

one_hot_lookup = [
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,0],
    [0,0,0,1]
]

x_one_hot = [one_hot_lookup[x] for x in x_data]

inputs = torch.Tensor(x_one_hot).view(seq_len, batch_size, input_size)
labels = torch.LongTensor(y_data) # 一维张量
# 没有用view转换为列向量，因为RNN的hidden输出是最后层每个hidden张量的组合张量

class Model(torch.nn.Module):
    def __init__(self,input_size,hidden_size,batch_size,num_layers):
        super(Model, self).__init__()
        self.number_layers = num_layers
        self.batch_size = batch_size
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.rnn = torch.nn.RNN(input_size = self.input_size, 
                                hidden_size = self.hidden_size,
                                num_layers = num_layers)
    
    def forward(self,input):
        hidden = torch.zeros(self.number_layers,
                             self.batch_size,
                             self.hidden_size)
        out,_ = self.rnn(input,hidden) # out是全部隐层输出
        return out.view(-1,self.hidden_size) # 转为（seq*batch，hidden）

net = Model(input_size,hidden_size,batch_size,num_layers)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)

In [ ]:
# 训练部分
for epoch in range(15):
    optimizer.zero_grad()
    outputs = net(inputs) # 得到的outputs即前馈中的out.view(seq*batch，hidden)
    loss = criterion(outputs, labels) 
    # 将out.view到2维的原因：criterion计算中，outputs和labels的形状要相同
    loss.backward()
    optimizer.step()

    _,idx = outputs.max(dim = 1) # 找最大hidden输出
    idx = idx.data.numpy()
    print('预测：',"".join([idx2char[x] for x in idx]), end = "")
    print(',epoch[%d/15] loss=%.3f' % (epoch+1, loss.item()))

预测： lloll,epoch[1/15] loss=1.176
预测： lloll,epoch[2/15] loss=1.176
预测： lloll,epoch[3/15] loss=1.176
预测： lloll,epoch[4/15] loss=1.176
预测： lloll,epoch[5/15] loss=1.176
预测： lloll,epoch[6/15] loss=1.176
预测： lloll,epoch[7/15] loss=1.176
预测： lloll,epoch[8/15] loss=1.176
预测： lloll,epoch[9/15] loss=1.176
预测： lloll,epoch[10/15] loss=1.176
预测： lloll,epoch[11/15] loss=1.176
预测： lloll,epoch[12/15] loss=1.176
预测： lloll,epoch[13/15] loss=1.176
预测： lloll,epoch[14/15] loss=1.176
预测： lloll,epoch[15/15] loss=1.176


## embedding优化one-hot

每个字符分配一个可训练的稠密向量

In [ ]:
num_class = 4 # 最终希望分类得到的类别数
input_size = 4 # 输入数据中有的字符种类数
hidden_size = 8
num_layers = 2
batch_size = 1
embedding_size = 10 # 规定每个字符由十个数表示

idx2char = ['e','h','l','o']
x_data = [[1,0,2,2,3]] # 
y_data = [3,1,2,3,2] # 

inputs = torch.LongTensor(x_data)
labels = torch.LongTensor(y_data)

class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.emb = torch.nn.Embedding(input_size, embedding_size)
        # 负责将维度转导embedding维
        self.rnn = torch.nn.RNN(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = torch.nn.Linear(hidden_size, num_class)

    def forward(self, x):
        hidden = torch.zeros(num_layers, x.size(0), hidden_size)
        # x.size(0)为batch_size
        x = self.emb(x)  # (batch, seqlen, embeddingSize)
        x, _ = self.rnn(x, hidden)
        x = self.fc(x)
        return x.view(-1, num_class)